# SHMS Bridge Anomaly Detection
## AI-based Anomaly Detection — Finger Bridge
**Phase 1 & 2: Data Loading, EDA, dan Preprocessing**

Dataset: 20 channel terpilih dari 90 channel (9 Accelerometer + 8 Cable + 2 Temperature + 1 Wind)

---

### 0. Import & Setup

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from shms_config import (
    ALL_CHANNELS, CHANNEL_ALIAS, ACCEL_CHANNELS, CABLE_CHANNELS,
    TEMP_CHANNELS, WIND_CHANNELS, CONTEXT_CHANNELS,
    OUTPUT_DIR, WINDOW_SIZE, WINDOW_STEP
)
from shms_phase1_loader import load_csv, check_sensor_health, print_health_summary, extract_selected
from shms_phase2_preprocessing import (
    AdaptiveNormalizer, compute_correlation_matrix,
    build_graph_edges, sliding_window, extract_stat_features
)

print('Setup OK')

### 1A. Load Data
Sesuaikan path file CSV kamu di bawah.

In [ ]:
# Sesuaikan path ke file CSV kamu
RAW_FILE      = 'data/raw/SAMPLE_20260329234000.csv'   # contoh
ABNORMAL_FILE = None   # isi jika punya file abnormal terpisah

df_raw = load_csv(RAW_FILE, is_abnormal=False)
print(f'Raw data: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} cols')
df_raw.head(3)

### 1B. Sensor Health Check
Diagnosis otomatis semua 90 channel.

In [ ]:
health = check_sensor_health(df_raw)
print_health_summary(health)

# Simpan laporan
health.to_csv(OUTPUT_DIR / 'p1_sensor_health.csv', index=False)
print('\nHealth report disimpan: output/p1_sensor_health.csv')

In [ ]:
# Ringkasan per status
health.groupby(['type','status'])['channel'].count().unstack(fill_value=0)

### 1C. Extract 20 Channel Terpilih

In [ ]:
df_sel = extract_selected(df_raw)
df_sel.to_csv(OUTPUT_DIR / 'p1_dataset_selected.csv', index=False)
df_sel.describe().T[['mean','std','min','max']]

### 1D. EDA — Correlation Matrix

In [ ]:
main_cols = [c for c in ALL_CHANNELS if c in df_sel.columns and c not in CONTEXT_CHANNELS]
corr_raw  = df_sel[main_cols].corr()
alias_map = {c: CHANNEL_ALIAS.get(c, c.replace('FB_','')) for c in main_cols}

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.eye(len(corr_raw), dtype=bool)
sns.heatmap(
    corr_raw.rename(index=alias_map, columns=alias_map),
    ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.3, annot=True, fmt='.2f',
    annot_kws={'size':7}, mask=mask, square=True
)
ax.set_title('Correlation matrix — sebelum normalisasi', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb_corr_raw.png', dpi=150, bbox_inches='tight')
plt.show()

### 1E. EDA — Time-series Preview per Kelompok

In [ ]:
groups = {
    'Accelerometer (Gal)': ACCEL_CHANNELS,
    'Cable Tension (kN)' : CABLE_CHANNELS,
    'Temperature (°C)'   : TEMP_CHANNELS,
}
colors = {'Accelerometer (Gal)':'#378ADD', 'Cable Tension (kN)':'#1D9E75', 'Temperature (°C)':'#D85A30'}

for grp, cols in groups.items():
    valid = [c for c in cols if c in df_sel.columns]
    if not valid: continue
    fig, axes = plt.subplots(len(valid), 1, figsize=(14, 2.2*len(valid)), sharex=True)
    if len(valid)==1: axes=[axes]
    for ax, col in zip(axes, valid):
        alias = CHANNEL_ALIAS.get(col, col)
        ax.plot(df_sel[col].values, color=colors[grp], linewidth=0.9)
        ax.axhline(df_sel[col].mean(), color='red', linewidth=0.7, linestyle='--', alpha=0.7)
        ax.set_ylabel(alias, fontsize=8, rotation=0, ha='right', labelpad=85)
        ax.grid(True, alpha=0.2)
    fig.suptitle(grp, fontsize=11)
    plt.tight_layout()
    plt.show()

---
### 2A. Adaptive Normalization
Z-score per channel, dihitung dari data normal (bukan dari data abnormal).
Stats disimpan ke CSV untuk reproducibility.

In [ ]:
ctx_cols = [c for c in CONTEXT_CHANNELS if c in df_sel.columns]

norm = AdaptiveNormalizer()
norm.fit(df_sel, cols=main_cols + ctx_cols)
df_norm = norm.fit_transform(df_sel, cols=main_cols + ctx_cols)
norm.save(OUTPUT_DIR / 'p2_normalizer_stats.csv')

# Cek hasil normalisasi
print('Statistik SEBELUM normalisasi:')
print(df_sel[main_cols[:5]].describe().T[['mean','std']].round(4))
print('\nStatistik SETELAH normalisasi:')
print(df_norm[main_cols[:5]].describe().T[['mean','std']].round(4))

### 2B. Correlation Matrix & Graph Edges untuk GNN

In [ ]:
corr_norm = compute_correlation_matrix(df_norm, cols=main_cols, save=True)
edges     = build_graph_edges(corr_norm, threshold=0.5)
print(f'\nTop 10 edges terkuat:')
print(edges.reindex(edges['weight'].abs().sort_values(ascending=False).index)
      .head(10)[['node_i','node_j','weight']].to_string(index=False))

### 2C. Sliding Window Segmentation
Window 1000 sampel = 10 detik @100Hz, overlap 50%.
> **Catatan**: untuk data sampel kecil, window otomatis dikecilkan.

In [ ]:
n = len(df_norm)
ws   = min(WINDOW_SIZE, max(20, n // 4))   # adaptif untuk data kecil
step = ws // 2
print(f'Data: {n} rows → window={ws}, step={step}')

X_all, y_all, meta = sliding_window(
    df_norm, cols=main_cols,
    window_size=ws, step=step,
    label_col='_is_abnormal' if '_is_abnormal' in df_norm.columns else None
)
print(f'X shape: {X_all.shape}  (windows × timesteps × channels)')

### 2D. Train/Test Split (time-ordered) & Statistical Features

In [ ]:
split   = max(1, int(len(X_all) * 0.8))
X_train, X_test = X_all[:split], X_all[split:]
y_train = y_all[:split] if y_all is not None else None
y_test  = y_all[split:] if y_all is not None else None

feat_train = extract_stat_features(X_train, main_cols)
feat_test  = extract_stat_features(X_test,  main_cols)

np.save(OUTPUT_DIR / 'p2_X_train.npy', X_train)
np.save(OUTPUT_DIR / 'p2_X_test.npy',  X_test)
if y_train is not None:
    np.save(OUTPUT_DIR / 'p2_y_train.npy', y_train)
    np.save(OUTPUT_DIR / 'p2_y_test.npy',  y_test)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'Feat train: {feat_train.shape}')
print('\nPhase 2 selesai → siap masuk Phase 3: Model Training')

---
## Ringkasan Output Phase 1 & 2

| File | Isi |
|---|---|
| `p1_sensor_health.csv` | Diagnosis semua 90 channel |
| `p1_dataset_selected.csv` | 20 channel terpilih, raw |
| `p2_normalizer_stats.csv` | Mean & std per channel (untuk deployment) |
| `p2_correlation_matrix.csv` | Correlation matrix 17×17 |
| `p2_graph_edges.csv` | Edge list untuk GNN |
| `p2_X_train.npy` | Array windows training (n, timestep, 17) |
| `p2_X_test.npy` | Array windows testing |
| `p2_stat_features.csv` | Fitur statistik per window (input Isolation Forest) |

**Next: Phase 3 — LSTM Autoencoder + GNN + Isolation Forest**